# Notebook 15 — C0: hyperparameter selection by held-out arbor reconstruction

Replaces the notebook-09 §S10 rule (*"max fingerprint silhouette among configs within 0.05 of the best causal R²"*), which selected on a **reported** metric — and, for the ViT and SqueezeNet, on silhouette alone, because `median_preact_r2` is `NaN` when a model has no fully connected layer. This notebook picks the per-layer rank with a criterion that never sees the fingerprint:

> **K\*(l) = the smallest rank whose *held-out* arbor reconstruction R² is within `PLATEAU_EPS` of the best held-out R² that layer reaches over `K ≤ K_CAP`.**

Held-out means the NMF basis is fitted on one stimulus split and scored on a disjoint one. That matters: **in-sample** arbor R² is monotone in K, so it has no interior optimum and any threshold on it is arbitrary — the old `K@R2.90` profile inherited that problem. Held-out R² can fall as K grows, so the criterion has a knee to find and cannot be gamed by over-ranking.

Three structural constraints are imposed on top of the criterion, so the trees are guaranteed to expose the structure the paper claims:

1. **The output layer's rank is large enough to hold every class** — `k_max[L] ≥ n_classes + LAST_EXTRA`, so the criterion chooses within a ceiling that is never the binding constraint. *(In the submitted runs it always was: CIFAR-10 and ImageNet both had `k_max[L] = 10` and both selected `K* = 10`, i.e. the rank was capped, not selected.)*
2. **Every output factor is traced** — `n_branches[L] = k_max[L]`, and `bft()` caps branching at `min(B, K*)`, so each recovered class circuit gets a child. No class is silently dropped. *(Submitted ImageNet followed 5 of 10 root factors, so only 5 of 8 super-categories had a traced circuit.)*
3. **Every output factor is split one layer back** — `n_branches[L-1] ≥ 2`, so the sub-structure claim rests on an actual split at every circuit, not just the ones that happened to be followed. A layer that factorizes into a single component spawns one child regardless, by the same `min(B, K*)` cap.

Also scales the two convolutional settings up (more stimuli, wider branching), per C0.4/C0.5 of `REBUTTAL_PLAN.md`.

**Out of scope here:** `stimulus_threshold` (τ) is *not* swept — it stays at each experiment's published value and is recorded. Sweeping τ and rank jointly is a second run; rank is the axis that was selected on the outcome metric.

**Silhouette and kNN are computed in §5 and recorded, but no selection step reads them.** They are there so the rebuttal can state what the old criterion would have chosen and by how much the answer moves.

## Pipeline

| § | step | cost |
|---|---|---|
| §2 | reference trace at a generous flat `k_max` (cheap branching) | 1 trace / model |
| §3 | per-layer held-out rank sweep on that trace's arbors | dominant cost |
| §4 | assemble the rank + branch profile under the three constraints | free |
| §5 | final trace at the selected profile; verify and diagnose | 1 trace / model |
| §6 | driver — runs §2–§5 for every model in `MODELS` | |
| §7 | summary table + `bft()` kwargs ready to paste into notebooks 01–05 | |

Every section checkpoints to `data/results/nb15_hp_<exp>.json` **after each layer of the sweep**, so a killed run keeps everything that finished. The driver skips models whose JSON is already complete unless `NB15_FORCE=1`.

## §0 · Setup

In [ ]:
import os, sys, json, gc, time, warnings
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import MiniBatchNMF

from src.bft import _safe_init

warnings.filterwarnings('ignore')
REPO   = os.path.abspath('..')                    # notebook lives in notebooks/
MODE   = os.environ.get('NB15_MODE', 'local')
MODE = "cluster"
FORCE  = os.environ.get('NB15_FORCE', '0') == '1'
N_JOBS = int(os.environ.get('NB15_JOBS', '3'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))

FIG_DIR = os.path.join(REPO, 'figs', '15_hp_selection')
RES_DIR = os.path.join(REPO, 'data', 'results')
for d in (FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

# ── compute profile ──────────────────────────────────────────────────────────
# SWEEP_ROWS caps the rows entering the rank sweep. The arbor's rank structure is a
# column-space property, so a few hundred rows resolve it; this decouples sweep cost
# from the (scaled-up) trace population.
if MODE == 'cluster':
    SWEEP_ROWS      = 900     # rows used in the rank sweep (fit + held-out together)
    SWEEP_MAX_ITER  = 300     # NMF iters inside the sweep
    TRACE_MAX_ITER  = 500     # NMF iters for the reference + final traces (bft default)
    MAX_NODES_LAYER = 2       # nodes pooled per layer in the sweep
    REF_ROOT_B      = 4       # root branching of the cheap REFERENCE trace
    STAB_SEEDS      = 5       # diagnostic only — never a selection criterion
else:
    SWEEP_ROWS, SWEEP_MAX_ITER, TRACE_MAX_ITER = 300, 60, 120
    MAX_NODES_LAYER, REF_ROOT_B, STAB_SEEDS = 1, 2, 2

HOLDOUT_FRAC = 0.30           # stimulus fraction held out of every NMF fit in the sweep
PLATEAU_EPS  = 0.01           # "within eps of the best held-out R2" -> the selection rule
R2_TARGETS   = (0.90, 0.95)   # recorded as alternative rules; not used for selection
SPLIT_SEED   = 0

# Optional: subsample columns of very wide arbors inside the SWEEP only (never in a
# trace). Rank structure survives random column subsampling; this is the knob that
# makes the SqueezeNet classifier arbor (512,000 columns) tractable if needed.
MAX_ARBOR_COLS = int(os.environ.get('NB15_MAXCOLS', '0')) or None

RNG = np.random.default_rng(SPLIT_SEED)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    return o if isinstance(o, (float, int, str)) or o is None else str(o)


class Recorder:
    """Per-model results with atomic checkpointing (nb09/nb14 convention)."""

    def __init__(self, exp):
        self.exp  = exp
        self.path = os.path.join(RES_DIR, f'nb15_hp_{exp}.json')
        self.results = {'experiment': exp, 'notebook': '15', 'mode': MODE,
                        'rule': ('K*(l) = smallest K whose HELD-OUT arbor reconstruction R2 '
                                 f'is within {PLATEAU_EPS} of that layer\'s best held-out R2 '
                                 f'(K <= K_CAP); holdout_frac={HOLDOUT_FRAC}. '
                                 'No fingerprint metric enters the selection.'),
                        'complete': False}
        self.completed = []

    def checkpoint(self, section=None):
        if section and section not in self.completed:
            self.completed.append(section)
        self.results['completed_sections'] = list(self.completed)
        tmp = self.path + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(jsonable(self.results), f, indent=1)
        os.replace(tmp, self.path)
        print(f'  [checkpoint] {self.exp}:{section or ""} -> '
              f'{os.path.relpath(self.path, REPO)} ({len(self.completed)} sections)')

    def done(self):
        self.results['complete'] = True
        self.checkpoint('DONE')


def savefig(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, bbox_inches='tight')
    print('  saved', os.path.relpath(p, REPO))
    plt.close(fig)
    return os.path.relpath(p, REPO)


print(f'MODE={MODE}  DEVICE={DEVICE}  SWEEP_ROWS={SWEEP_ROWS}  '
      f'MAX_ARBOR_COLS={MAX_ARBOR_COLS}  FORCE={FORCE}')

## §1 · **Select the models to run here**

`MODELS` is the switch. Either edit the list, or set `NB15_MODELS` as a comma-separated
env var (`NB15_MODELS=cnn_cifar,imagenet_cnn`), which overrides the list.

`SCALE` carries the per-model scale-up and the structural constraints:

| key | meaning |
|---|---|
| `conf_per_class` | stimuli kept per class/category (`None` = leave the nb09 default). **This is the C0.4/C0.5 scale-up.** |
| `k_cap` | ceiling of the rank sweep — the criterion must be free to pick below it |
| `last_extra` | output-layer `k_max` floor is `n_classes + last_extra` (constraint 1) |
| `b_second_last` | branching one layer before the output, `≥ 2` (constraint 3) |

Constraint 2 (`n_branches[L] = k_max[L]`) is not a knob — it is applied unconditionally.

In [ ]:
# ── the models this run covers ───────────────────────────────────────────────
MODELS = [
    #'mlp_even_odd',      # SimpleMLP 784->8->4->2, MNIST {0,1,3,4} even/odd
    #'mlp_digit',         # SimpleMLP 784->40->20->10, MNIST 10-way
    'cnn_cifar',         # SmallCNN, CIFAR-10
    #'vit_mnist',         # TinyViT, MNIST even/odd
    'imagenet_cnn',      # SqueezeNet 1.1 (pretrained), ImageNet 8 super-categories
]
if os.environ.get('NB15_MODELS'):
    MODELS = [m.strip() for m in os.environ['NB15_MODELS'].split(',') if m.strip()]

# ── scale-up + structural constraints, per model ─────────────────────────────
SCALE = {
    #                 stimuli/class   sweep ceiling   output floor   split branching
    'mlp_even_odd': dict(conf_per_class=None, k_cap=8,  last_extra=2, b_second_last=2),
    'mlp_digit':    dict(conf_per_class=None, k_cap=16, last_extra=4, b_second_last=2),
    'cnn_cifar':    dict(conf_per_class=200,  k_cap=16, last_extra=4, b_second_last=3),
    'vit_mnist':    dict(conf_per_class=None, k_cap=10, last_extra=2, b_second_last=2),
    # SqueezeNet keeps b_second_last=2 (the required minimum): its root branching already
    # rises 5 -> n_classes+4 = 12, so every super-category gets a circuit, and 3 would
    # roughly double an already 6-12 h run. Raise it if the budget allows.
    'imagenet_cnn': dict(conf_per_class=250,  k_cap=16, last_extra=4, b_second_last=2),
}
# Submitted populations for reference: cnn_cifar 60/class (S=600),
# imagenet_cnn 50/class in nb09 and 100/class in the paper figures (S=544).
# The MLPs and the ViT already trace their full correct test split, so they do not scale.

print('models to run:', MODELS)
for m in MODELS:
    s = SCALE[m]
    print(f'  {m:14s} conf/class={str(s["conf_per_class"]):>4s}  k_cap={s["k_cap"]:2d}  '
          f'last_extra={s["last_extra"]}  b_second_last={s["b_second_last"]}')

## §2 · Inherit notebook 09's pipeline

Same device as notebook 10: this notebook does **not** redefine model loading, stimulus
selection or arbor construction. It `exec`s notebook 09's §0 (setup + helpers), §1
(registry) and §2 (`build_experiment`) into a **fresh namespace per model**, so a profile
selected here is directly comparable to everything nb09 measured, and so the five models
cannot contaminate each other's globals.

The scale-up and the reference-trace hyperparameters are injected into `REG` *between* §1
and §2 — i.e. after the registry exists and before `build_experiment` reads it.

The **reference trace** deliberately uses cheap branching (`REF_ROOT_B` root factors, 2 one
layer back). Its only job is to supply, for each layer, arbors carrying realistic
stimulus weighting; the sweep pools up to `MAX_NODES_LAYER` nodes per layer. The full
structural tree is built once, in §5, at the selected profile.

In [ ]:
import nbformat

_NB09 = '09_validation_all_models.ipynb'
_nb09 = nbformat.read(_NB09, as_version=4)
_SETUP = []
for _c in _nb09.cells:
    if _c.cell_type != 'code':
        continue
    _SETUP.append(_c.source)
    if 'build_experiment(EXP)' in _c.source:      # §2's last line — stop there
        break
else:
    raise RuntimeError('nb09 setup cells not found: no cell calls build_experiment(EXP).')
print(f'inherited {len(_SETUP)} setup cells from {_NB09} '
      f'(§0 helpers, §1 registry, §2 build_experiment)')


def load_ctx(exp):
    """Build one model's context in an isolated namespace, with C0 overrides applied.

    Returns the namespace dict; `ns['ctx']`, `ns['tree']` (the reference trace),
    `ns['nodes_by_layer']`, `ns['arbor_pos']`, `ns['targets']` etc. are nb09's own objects.
    """
    os.environ['NB09_EXP'], os.environ['NB09_MODE'] = exp, MODE
    ns = {'__name__': f'nb09_ctx_{exp}', '__builtins__': __builtins__}

    for i, src in enumerate(_SETUP[:-1]):                 # §0 + §1
        exec(compile(src, f'{_NB09}:setup{i}', 'exec'), ns)

    # ── C0 overrides, applied to the registry before build_experiment reads it ──
    r, sc = ns['REG'][exp], SCALE[exp]
    n_layers = len(r['bft']['k_max'])
    if sc['conf_per_class'] is not None and 'conf_per_class' in r:
        r['conf_per_class'] = sc['conf_per_class']
    # Reference trace: flat generous rank, cheap branching.
    ref_b = [1] * (n_layers - 2) + [2, REF_ROOT_B] if n_layers >= 2 else [REF_ROOT_B]
    r['bft'] = dict(r['bft'])
    r['bft']['k_max'] = [sc['k_cap']] * n_layers
    r['bft']['n_branches'] = ref_b
    ns['BFT_MAX_ITER'] = TRACE_MAX_ITER
    ns['N_TRACE'] = None if MODE == 'cluster' else ns['N_TRACE']

    exec(compile(_SETUP[-1], f'{_NB09}:setup_build', 'exec'), ns)      # §2
    return ns


def release(ns):
    """Drop a model's namespace and free GPU memory before the next one."""
    ns.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## §3 · Held-out rank sweep

For every layer, for every candidate rank `K`:

1. split the node's stimuli into a fit set and a disjoint held-out set (`HOLDOUT_FRAC`,
   stratified by class, one split reused for every layer and every `K` so the curves are
   paired);
2. fit NMF on the fit rows → basis `H`;
3. project the **held-out** rows onto that fixed `H` (`MiniBatchNMF.transform`, the same
   fixed-basis non-negative solve the paper's NNLS projection performs) and score
   `R² = 1 − ‖X_ho − W_ho Hᵀ‖² / ‖X_ho − mean‖²`.

In-sample R² is recorded next to it. The gap between the two curves is the figure that
justifies the whole change of rule: in-sample rises monotonically with `K`, held-out
turns over.

Wide layers get a coarse rank grid (and, if `NB15_MAXCOLS` is set, a random column
subsample) — both recorded per layer.

In [ ]:
def _mb_nmf(X, K, max_iter):
    """MiniBatchNMF with src.bft's solver settings (deterministic nndsvda where legal)."""
    n_s, n_f = X.shape
    return MiniBatchNMF(n_components=K, init=_safe_init(K, n_s, n_f), random_state=0,
                        max_iter=max_iter, batch_size=max(64, min(1024, n_s // 4)),
                        tol=1e-3, max_no_improvement=5, l1_ratio=0)


def k_grid(cap, n_cols, wide_thresh=100_000):
    """1..cap for ordinary layers; a coarse grid for very wide arbors."""
    if n_cols < wide_thresh:
        return list(range(1, cap + 1))
    coarse = [1, 2, 3, 4, 6, 8, 10, 12, 14, 16, 20, 24]
    return [k for k in coarse if k <= cap] or [cap]


def stratified_split(y, frac, seed=SPLIT_SEED):
    """Disjoint (fit, holdout) row indices, stratified so every class appears in both."""
    rng = np.random.default_rng(seed)
    fit, ho = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n_ho = max(1, int(round(frac * len(idx)))) if len(idx) > 1 else 0
        ho.extend(idx[:n_ho]); fit.extend(idx[n_ho:])
    return np.sort(np.array(fit, int)), np.sort(np.array(ho, int))


def heldout_curve(X, y, cap, max_iter):
    """Per-K in-sample and held-out arbor reconstruction on one node's positive arbor."""
    X = np.ascontiguousarray(X, dtype=np.float32)
    if MAX_ARBOR_COLS and X.shape[1] > MAX_ARBOR_COLS:
        cols = np.sort(RNG.choice(X.shape[1], MAX_ARBOR_COLS, replace=False))
        X, sub_cols = X[:, cols], int(MAX_ARBOR_COLS)
    else:
        sub_cols = None
    if X.shape[0] > SWEEP_ROWS:                       # cap rows, keeping class balance
        keep, _ = stratified_split(y, 1.0 - SWEEP_ROWS / X.shape[0])
        X, y = X[keep], y[keep]

    fit, ho = stratified_split(y, HOLDOUT_FRAC)
    if len(ho) < 4 or len(fit) < 4:
        return [], {'sub_cols': sub_cols, 'n_fit': len(fit), 'n_ho': len(ho),
                    'skipped': 'too few rows to split'}
    Xf, Xh = X[fit], X[ho]
    ss_tot_h = float(((Xh - Xh.mean()) ** 2).sum())
    ss_tot_f = float(((Xf - Xf.mean()) ** 2).sum())
    rows = []
    for K in k_grid(min(cap, min(Xf.shape) - 1), X.shape[1]):
        if K < 1:
            continue
        m = _mb_nmf(Xf, K, max_iter)
        Wf = m.fit_transform(Xf)
        Wh = m.transform(Xh)
        res_h = float(((Xh - Wh @ m.components_) ** 2).sum())
        res_f = float(((Xf - Wf @ m.components_) ** 2).sum())
        rows.append({'K': int(K),
                     'heldout_r2':  1.0 - res_h / ss_tot_h if ss_tot_h > 0 else float('nan'),
                     'insample_r2': 1.0 - res_f / ss_tot_f if ss_tot_f > 0 else float('nan'),
                     'heldout_rel_err': float(np.sqrt(res_h) /
                                              (np.linalg.norm(Xh) + 1e-12))})
    return rows, {'sub_cols': sub_cols, 'n_fit': int(len(fit)), 'n_ho': int(len(ho)),
                  'n_cols': int(X.shape[1])}


def pool_curves(curves):
    """Average several nodes' curves at each K (only Ks every node reached)."""
    if not curves:
        return []
    common = sorted(set.intersection(*[{r['K'] for r in c} for c in curves]))
    out = []
    for K in common:
        vals = [[r for r in c if r['K'] == K][0] for c in curves]
        out.append({'K': K,
                    'heldout_r2':  float(np.mean([v['heldout_r2'] for v in vals])),
                    'insample_r2': float(np.mean([v['insample_r2'] for v in vals])),
                    'heldout_rel_err': float(np.mean([v['heldout_rel_err'] for v in vals])),
                    'n_nodes': len(vals)})
    return out


def pick_K(curve):
    """THE selection rule + the alternatives + the ceiling diagnostics, all recorded.

    primary  : smallest K within PLATEAU_EPS of the best held-out R2  (always exists)
    argmax   : K maximising held-out R2
    at_r2_XX : smallest K reaching an absolute held-out R2 target (None if never)

    Two independent signals that the sweep ceiling `k_cap` was too low:
      selected_at_sweep_cap : the rule picked the largest K on the grid
      still_rising          : the last rank step still bought more than PLATEAU_EPS of
                              held-out R2, so the curve had not plateaued by the ceiling.
    Either one means the number is a ceiling, not a selection — raise k_cap and re-run.
    """
    if not curve:
        return {'K': None, 'reason': 'empty curve'}
    r2 = np.array([r['heldout_r2'] for r in curve], float)
    Ks = np.array([r['K'] for r in curve], int)
    best = float(np.nanmax(r2))
    within = Ks[r2 >= best - PLATEAU_EPS]
    out = {'K': int(within.min()), 'best_heldout_r2': best,
           'K_argmax': int(Ks[int(np.nanargmax(r2))]),
           'selected_at_sweep_cap': bool(int(within.min()) == int(Ks.max())),
           'still_rising': bool(len(r2) >= 2 and (r2[-1] - r2[-2]) > PLATEAU_EPS),
           'reason': f'smallest K within {PLATEAU_EPS} of best held-out R2 {best:.4f}'}
    for t in R2_TARGETS:
        hit = Ks[r2 >= t]
        out[f'K_at_r2_{int(t * 100)}'] = int(hit.min()) if len(hit) else None
    return out


def sweep_layers(ns, rec, cap):
    """Held-out rank sweep over every traced layer; checkpoints after each layer."""
    nodes_by_layer, tree = ns['nodes_by_layer'], ns['tree']
    targets = ns['targets'].astype(int)
    per_layer_nodes = {}
    for nd in tree.nodes():
        per_layer_nodes.setdefault(nd.layer_idx, []).append(nd)

    sweep = {}
    rec.results['sweep'] = sweep
    for li in sorted(nodes_by_layer):
        t0 = time.perf_counter()
        nds = per_layer_nodes[li][:MAX_NODES_LAYER]
        curves, meta = [], []
        for nd in nds:
            X = ns['node_arbor_pos'](nd)
            # A node's arbor rows are the full population; rows the tau-gate zeroed carry
            # no signal, so drop them before splitting.
            keep = np.where(np.abs(X).sum(1) > 0)[0]
            if len(keep) < 8:
                continue
            c, mt = heldout_curve(X[keep], targets[keep], cap, SWEEP_MAX_ITER)
            del X
            gc.collect()
            if c:
                curves.append(c); meta.append(mt)
        pooled = pool_curves(curves)
        sel = pick_K(pooled)
        sweep[str(li)] = {'layer_idx': int(li),
                          'layer_name': nodes_by_layer[li].layer_name,
                          'layer_type': nodes_by_layer[li].layer_type,
                          'n_nodes_pooled': len(curves), 'meta': meta,
                          'curve': pooled, 'selection': sel,
                          'wall_s': round(time.perf_counter() - t0, 1)}
        print(f'  L{li} {nodes_by_layer[li].layer_name!r:16s} '
              f'K*={sel.get("K")}  argmax={sel.get("K_argmax")}  '
              f'bestR2_ho={sel.get("best_heldout_r2", float("nan")):.4f}  '
              f'K@R2.90={sel.get("K_at_r2_90")}  ({sweep[str(li)]["wall_s"]:.0f}s)')
        rec.checkpoint(f'sweep_L{li}')
    return sweep

## §4 · Assemble the profile

The three structural constraints are applied here, each recorded separately so the JSON
shows exactly where the criterion was overridden and by how much.

In [ ]:
def build_profile(sweep, n_classes, sc, layer_ids):
    """Turn the per-layer selections into (k_max, n_branches), applying the constraints.

    layer_ids are forward order (0 = input side), matching bft()'s list arguments.
    """
    k_sel = [sweep[str(li)]['selection'].get('K') for li in layer_ids]
    k_sel = [int(k) if k else 1 for k in k_sel]
    k_max = list(k_sel)
    notes = []

    # constraint 1 — the output layer must be able to hold every class, plus headroom
    L = len(layer_ids) - 1
    floor = int(n_classes + sc['last_extra'])
    output_floor_applied = k_max[L] < floor
    if output_floor_applied:
        notes.append(f'output k_max raised {k_max[L]} -> {floor} '
                     f'(n_classes {n_classes} + last_extra {sc["last_extra"]})')
        k_max[L] = floor
    else:
        notes.append(f'output k_max {k_max[L]} came from the criterion '
                     f'(already >= floor {floor})')

    # constraint 2 — trace EVERY output factor (bft caps at min(B, K*))
    n_branches = [1] * len(layer_ids)
    n_branches[L] = k_max[L]
    notes.append(f'n_branches[out] = k_max[out] = {k_max[L]} (every output factor traced)')

    # constraint 3 — split every circuit one layer back
    if L - 1 >= 0:
        n_branches[L - 1] = max(2, int(sc['b_second_last']))
        notes.append(f'n_branches[out-1] = {n_branches[L - 1]} '
                     f'(>= 2: every circuit splits, unless K*=1 there)')

    # A layer is ceiling-bound when the CRITERION ran out of grid, not when K* == k_max
    # (which is expected everywhere, since k_max is set FROM the criterion).
    capped = [li for li in layer_ids
              if sweep[str(li)]['selection'].get('selected_at_sweep_cap')
              or sweep[str(li)]['selection'].get('still_rising')]
    if capped:
        notes.append(f'WARNING: the held-out curve had not plateaued by the sweep ceiling '
                     f'in layers {capped} — raise k_cap ({sc["k_cap"]}) and re-run those; '
                     f'their K* is a ceiling, not a selection')
    return {'k_max': k_max, 'n_branches': n_branches,
            'k_from_criterion': k_sel, 'notes': notes, 'k_cap_hit': capped,
            'output_floor_applied': bool(output_floor_applied)}


def est_nodes(n_branches):
    """Node count of the tree a branch vector induces (upper bound; K* may cap it)."""
    n, level = 1, 1
    for b in reversed(n_branches[1:]):        # root layer spawns children with its own B
        level *= max(1, int(b))
        n += level
    return n

## §5 · Final trace at the selected profile — verification

One trace at the selected `(k_max, n_branches)`, then:

- **held-out arbor R² per node** at the rank the trace actually used — the criterion,
  re-measured on the real tree rather than the reference one;
- **`K*` vs `k_max` per layer** — flags any layer where the rank is still *capped* rather
  than selected (that is a signal to raise `k_cap`, not a result);
- **class coverage at the output** — how many distinct classes are the argmax of some
  output factor, and which classes no factor claims. Constraint 1 exists to make this
  come out complete;
- **NMF stability** across random initialisations (diagnostic);
- **fingerprint silhouette / kNN** — *recorded, not used for selection*, so the rebuttal
  can quantify how far the answer moves from the old rule.

In [ ]:
def verify_profile(ns, rec, profile, sc):
    """Run the final trace and record every diagnostic. Selection is already fixed."""
    from src import extract_fingerprint_matrix

    ctx, targets = ns['ctx'], ns['targets'].astype(int)
    t0 = time.perf_counter()
    tree = ns['ctx']['bft_call'](k_max=profile['k_max'],
                                 n_branches=profile['n_branches'])
    wall = time.perf_counter() - t0
    nodes = list(tree.nodes())
    by_layer = {}
    for nd in nodes:
        by_layer.setdefault(nd.layer_idx, []).append(nd)
    layer_ids = sorted(by_layer)

    # ── K* actually selected vs the ceiling it was given ─────────────────────
    # k_max comes FROM the criterion at every layer except the output (raised to the
    # class floor), so K* == k_max is the expected outcome, not a warning. The
    # actionable flag is `ceiling_bound`: K* is at k_max AND that k_max was imposed
    # (output floor) or the sweep never plateaued at that layer.
    L_out = max(layer_ids)
    ranks = {}
    for li in layer_ids:
        ks = [int(nd.img_factors.shape[1]) for nd in by_layer[li]]
        at_kmax = bool(max(ks) >= profile['k_max'][li])
        imposed = ((li == L_out and profile.get('output_floor_applied'))
                   or (li in profile['k_cap_hit']))
        ranks[str(li)] = {'k_max': int(profile['k_max'][li]),
                          'k_star_min': min(ks), 'k_star_max': max(ks),
                          'k_star_equals_kmax': at_kmax,
                          'ceiling_bound': bool(at_kmax and imposed)}

    # ── held-out arbor R2 on the FINAL tree, at the rank it used ─────────────
    ho = {}
    for li in layer_ids:
        vals = []
        for nd in by_layer[li][:MAX_NODES_LAYER]:
            X = ns['node_arbor_pos'](nd)
            keep = np.where(np.abs(X).sum(1) > 0)[0]
            if len(keep) < 8:
                continue
            c, _ = heldout_curve(X[keep], targets[keep], int(nd.img_factors.shape[1]),
                                 SWEEP_MAX_ITER)
            del X; gc.collect()
            hit = [r for r in c if r['K'] == int(nd.img_factors.shape[1])]
            if hit:
                vals.append(hit[0]['heldout_r2'])
        ho[str(li)] = float(np.mean(vals)) if vals else float('nan')

    # ── class coverage at the output ─────────────────────────────────────────
    root = tree.root
    W = np.asarray(root.img_factors, float)
    classes = np.unique(targets)
    prof = np.stack([W[targets == c].mean(0) for c in classes])          # (C, K)
    prof = prof / (prof.sum(0, keepdims=True) + 1e-12)
    owner = classes[prof.argmax(0)]
    covered = sorted(set(int(o) for o in owner))
    missing = [int(c) for c in classes if int(c) not in covered]
    coverage = {'n_output_factors': int(W.shape[1]),
                'factor_to_class': [int(o) for o in owner],
                'classes_covered': covered, 'classes_missing': missing,
                'complete': len(missing) == 0}

    # ── diagnostics that are NOT selection criteria ──────────────────────────
    stab = {}
    for li in layer_ids:
        nd = by_layer[li][0]
        try:
            s = ns['stab_mean'](ns['node_arbor_pos'](nd),
                                int(nd.img_factors.shape[1]), STAB_SEEDS)
            stab[str(li)] = float(np.mean(s))
        except Exception as e:
            stab[str(li)] = float('nan')
    n_rows = tree.root.img_factors.shape[0]
    F = extract_fingerprint_matrix(tree, np.arange(n_rows))
    y = targets if len(targets) == n_rows else np.asarray(tree.targets).astype(int)
    sil, knn = ns['sep_metrics'](F, y)

    out = {'wall_s': round(wall, 1), 'n_nodes': len(nodes),
           'n_samples': int(n_rows), 'ranks': ranks,
           'heldout_r2_final': ho, 'class_coverage': coverage,
           'nmf_stability': stab,
           'NOT_A_CRITERION': {'note': 'recorded for the rebuttal only; no selection '
                                       'step in this notebook reads these',
                               'fingerprint_silhouette': sil,
                               'fingerprint_knn_acc': knn,
                               'fingerprint_dim': int(F.shape[1])}}
    print(f'  final trace: {len(nodes)} nodes, {wall / 60:.1f} min, fp dim {F.shape[1]}')
    print(f'    ranks (k*/k_max): ' +
          '  '.join(f'L{li}:{ranks[str(li)]["k_star_max"]}/{ranks[str(li)]["k_max"]}'
                    + ('!' if ranks[str(li)]['ceiling_bound'] else '')
                    for li in layer_ids) + '   (! = ceiling-bound)')
    print(f'    output covers {len(covered)}/{len(classes)} classes'
          + (f' — MISSING {missing}' if missing else ' (complete)'))
    print(f'    [not a criterion] silhouette={sil:.3f}  knn={knn:.3f}')
    return out, tree


def plot_model(exp, sweep, profile, layer_ids):
    """(a) held-out vs in-sample R2 per layer, (b) the selected profile."""
    n = len(layer_ids)
    fig, axes = plt.subplots(1, n + 1, figsize=(2.5 * (n + 1), 2.9))
    for ax, li in zip(axes[:-1], layer_ids):
        c = sweep[str(li)]['curve']
        if not c:
            ax.set_axis_off(); continue
        K = [r['K'] for r in c]
        ax.plot(K, [r['insample_r2'] for r in c], 'o-', ms=3, c='0.6', label='in-sample')
        ax.plot(K, [r['heldout_r2'] for r in c], 'o-', ms=3, c='#e15759', label='held-out')
        ks = sweep[str(li)]['selection'].get('K')
        if ks:
            ax.axvline(ks, ls=':', c='#4e79a7')
        ax.set_title(f'L{li} {sweep[str(li)]["layer_name"]}', fontsize=7)
        ax.set_xlabel('K', fontsize=7); ax.tick_params(labelsize=6)
        ax.set_ylim(min(0, min(r['heldout_r2'] for r in c)) - 0.02, 1.02)
    axes[0].set_ylabel('arbor $R^2$', fontsize=7); axes[0].legend(fontsize=6)
    ax = axes[-1]
    x = np.arange(len(layer_ids))
    ax.bar(x - 0.2, profile['k_from_criterion'], 0.4, label='held-out $K^*$', color='#e15759')
    ax.bar(x + 0.2, profile['k_max'], 0.4, label='$k_{max}$ used', color='#4e79a7')
    ax.plot(x, profile['n_branches'], 'k^--', ms=4, label='branches')
    ax.set_xticks(x); ax.set_xticklabels([f'L{li}' for li in layer_ids], fontsize=6)
    ax.set_title('selected profile', fontsize=7); ax.legend(fontsize=6)
    ax.tick_params(labelsize=6)
    fig.suptitle(f'nb15 C0 — {exp}  (dotted = selected K; held-out turns over, '
                 f'in-sample does not)', y=1.06, fontsize=9)
    fig.tight_layout()
    return savefig(fig, f'fig_hp_{exp}.pdf')

## §6 · Driver

One model at a time; the namespace is released between models so peak memory is one
model's arbors, not five. A model whose JSON is already `complete` is skipped unless
`NB15_FORCE=1`.

In [ ]:
SUMMARY = {}

for exp in MODELS:
    rec = Recorder(exp)
    if os.path.exists(rec.path) and not FORCE:
        try:
            prev = json.load(open(rec.path))
            if prev.get('complete'):
                print(f'\n=== {exp}: already complete — skipping (NB15_FORCE=1 to redo) ===')
                SUMMARY[exp] = prev
                continue
        except Exception:
            pass

    print(f'\n{"=" * 78}\n=== {exp} ===\n{"=" * 78}')
    sc = SCALE[exp]
    t_model = time.perf_counter()
    try:
        ns = load_ctx(exp)
    except Exception as e:
        print(f'  SKIPPED — could not build context: {e}')
        rec.results['error'] = str(e)
        rec.checkpoint('error')
        continue

    layer_ids = sorted(ns['nodes_by_layer'])
    n_classes = int(ns['ctx']['n_classes'])
    rec.results.update(
        n_samples=int(ns['n_samples']), n_classes=n_classes, n_layers=len(layer_ids),
        layer_names=[ns['nodes_by_layer'][li].layer_name for li in layer_ids],
        scale=jsonable(sc),
        stimulus_threshold=float(ns['ctx']['bft_kwargs'].get('stimulus_threshold', 0.0)),
        reference_trace=jsonable(ns['ctx']['bft_kwargs']),
        config=dict(sweep_rows=SWEEP_ROWS, sweep_max_iter=SWEEP_MAX_ITER,
                    trace_max_iter=TRACE_MAX_ITER, holdout_frac=HOLDOUT_FRAC,
                    plateau_eps=PLATEAU_EPS, max_arbor_cols=MAX_ARBOR_COLS,
                    max_nodes_layer=MAX_NODES_LAYER, split_seed=SPLIT_SEED))
    print(f'  context: n_samples={ns["n_samples"]}  n_classes={n_classes}  '
          f'layers={len(layer_ids)}  tau={rec.results["stimulus_threshold"]}')
    rec.checkpoint('context')

    # §3 — held-out rank sweep
    sweep = sweep_layers(ns, rec, sc['k_cap'])

    # §4 — profile under the structural constraints
    profile = build_profile(sweep, n_classes, sc, layer_ids)
    profile['n_nodes_estimate'] = est_nodes(profile['n_branches'])
    rec.results['profile'] = profile
    print('  profile: k_max=%s  n_branches=%s  (<= %d nodes)'
          % (profile['k_max'], profile['n_branches'], profile['n_nodes_estimate']))
    for nline in profile['notes']:
        print('    -', nline)
    rec.checkpoint('profile')

    # §5 — final trace + verification
    try:
        ver, final_tree = verify_profile(ns, rec, profile, sc)
        rec.results['verification'] = ver
        rec.results['final_bft_kwargs'] = jsonable({
            'k_max': profile['k_max'], 'n_branches': profile['n_branches'],
            'stimulus_threshold': rec.results['stimulus_threshold'],
            'weighting': 'img_selectivity', 'normalization': 'none',
            **({'conf_per_class': sc['conf_per_class']} if sc['conf_per_class'] else {}),
            **({'conv_pool_method': 'avg'}
               if ns['ctx']['bft_kwargs'].get('conv_pool_method') else {})})
        rec.checkpoint('verification')
        del final_tree
    except Exception as e:
        print(f'  verification FAILED: {e}')
        rec.results['verification_error'] = str(e)
        rec.checkpoint('verification_error')

    try:
        rec.results['figure'] = plot_model(exp, sweep, profile, layer_ids)
    except Exception as e:
        print(f'  figure failed: {e}')

    rec.results['wall_s_total'] = round(time.perf_counter() - t_model, 1)
    rec.done()
    SUMMARY[exp] = rec.results
    release(ns)
    print(f'  {exp} done in {rec.results["wall_s_total"] / 60:.1f} min')

## §7 · Summary — the profiles, ready to paste

Prints one row per model and writes `data/results/nb15_hp_summary.json`. The
`final_bft_kwargs` block for each model is what notebooks 01–05 (and the C1/C2 runs)
should adopt.

Read the **`ceilbnd`** column first. `K* == k_max` is *expected* at most layers — `k_max`
is set **from** the criterion — so that alone is not a warning. A `!` marks the case that
is: the rank is bound by a ceiling the criterion did not choose, either the output-layer
class floor or a layer whose held-out curve had not plateaued by `k_cap`. Raise that
model's `k_cap` / `last_extra` and re-run it with `NB15_MODELS=<exp> NB15_FORCE=1`.

In [ ]:
rows = []
for exp in MODELS:
    r = SUMMARY.get(exp)
    if not r or 'profile' not in r:
        print(f'{exp:14s}  (no result)')
        continue
    p, v = r['profile'], r.get('verification', {})
    ranks = v.get('ranks', {})
    cap = ''.join('!' if ranks.get(str(i), {}).get('ceiling_bound') else '.'
                  for i in range(r['n_layers']))
    cov = v.get('class_coverage', {})
    rows.append(dict(exp=exp, n=r['n_samples'], k_max=p['k_max'],
                     B=p['n_branches'], nodes=v.get('n_nodes'), capped=cap,
                     covered=f"{len(cov.get('classes_covered', []))}/{r['n_classes']}",
                     ho_r2=np.nanmean([float(x) for x in
                                       v.get('heldout_r2_final', {}).values()])
                     if v.get('heldout_r2_final') else float('nan'),
                     sil=v.get('NOT_A_CRITERION', {}).get('fingerprint_silhouette'),
                     mins=round(r.get('wall_s_total', 0) / 60, 1)))

print(f'{"model":14s} {"S":>6s} {"nodes":>6s} {"ceilbnd":>8s} {"cls":>6s} '
      f'{"hoR2":>6s} {"sil*":>6s} {"min":>6s}   k_max / branches')
print('-' * 110)
for d in rows:
    print(f'{d["exp"]:14s} {d["n"]:6d} {str(d["nodes"]):>6s} {d["capped"]:>8s} '
          f'{d["covered"]:>6s} {d["ho_r2"]:6.3f} '
          f'{(d["sil"] if d["sil"] is not None else float("nan")):6.3f} {d["mins"]:6.1f}   '
          f'{d["k_max"]} / {d["B"]}')
print('\n* silhouette is reported only; it is NOT part of the selection rule.')
print('  ceilbnd: "!" = that layer\'s rank is bound by an imposed ceiling, not chosen by')
print('           the criterion (output class floor, or the sweep never plateaued).')
print('           Raise k_cap / last_extra for that model and re-run with NB15_FORCE=1.')

print('\n' + '=' * 78 + '\nfinal_bft_kwargs — paste into notebooks 01-05 / C1 / C2\n'
      + '=' * 78)
for exp in MODELS:
    r = SUMMARY.get(exp)
    if r and 'final_bft_kwargs' in r:
        print(f'\n{exp}:\n  {json.dumps(r["final_bft_kwargs"])}')

out = os.path.join(RES_DIR, 'nb15_hp_summary.json')
with open(out, 'w') as f:
    json.dump(jsonable({'mode': MODE, 'models': MODELS, 'table': rows,
                        'final_bft_kwargs': {e: SUMMARY[e].get('final_bft_kwargs')
                                             for e in SUMMARY},
                        'per_model': {e: {k: v for k, v in SUMMARY[e].items()
                                          if k != 'sweep'} for e in SUMMARY}}), f, indent=1)
print('\nwrote', os.path.relpath(out, REPO))

## How to run on the cluster

`NB15_MODE=cluster` lifts the laptop caps (900 sweep rows, 300/500 NMF iters, 2 nodes
pooled per layer, 5 stability seeds, full trace populations).

```bash
cd notebooks

# everything
NB15_MODE=cluster \
  ../.venv/bin/python -m nbconvert --to notebook --execute \
  --output executed_15_hp_selection.ipynb \
  --ExecutePreprocessor.timeout=400000 15_hp_selection.ipynb

# or one model per job — recommended, the two conv models dominate the wall time
NB15_MODE=cluster NB15_MODELS=imagenet_cnn \
  ../.venv/bin/python -m nbconvert --to notebook --execute \
  --output executed_15_hp_imagenet.ipynb \
  --ExecutePreprocessor.timeout=400000 15_hp_selection.ipynb
```

**Environment variables**

| var | default | meaning |
|---|---|---|
| `NB15_MODE` | `local` | `cluster` for the full profile |
| `NB15_MODELS` | all five | comma-separated subset, overrides the `MODELS` list |
| `NB15_FORCE` | `0` | `1` re-runs models whose JSON is already `complete` |
| `NB15_MAXCOLS` | unset | random column subsample for the **sweep only**; `150000` cuts SqueezeNet sweep time ~3× |
| `NB15_JOBS` | `3` | BLAS threads |

**Needs** (same as nb09/nb14): `mnist_even_odd_mlp_8_4_0134_seed0`,
`mnist_digit_mlp_40_20_seed0`, `cifar10_cnn_seed0`, `mnist_even_odd_vit_tiny_seed0` under
`data/models/`, and ImageNet val images under `data/` (torchvision `ImageNet` root or an
`ImageFolder` at `data/val`). MNIST and CIFAR-10 download themselves. A model whose
checkpoint or data is missing is skipped with a recorded error, not a crash.

**Rough GPU budget.** MLPs and the ViT: minutes. CIFAR CNN at 200 stimuli/class: 1–3 h
(the sweep dominates; the final trace is ~80 nodes). SqueezeNet at 250/category: 6–12 h —
the classifier arbor is 512,000 columns wide, and the final tree is ~250 nodes at
`b_second_last=3`. Set `NB15_MAXCOLS=150000` and/or drop `b_second_last` to 2 if that is
too long.

**Output**

- `data/results/nb15_hp_<exp>.json` — one per model. Carries the full per-layer held-out
  curve, the selection and every alternative rule, the assembled profile with the
  constraint notes, and the verification block. Checkpointed after **every layer**, so a
  killed run keeps its finished layers.
- `data/results/nb15_hp_summary.json` — the table plus `final_bft_kwargs` per model.
- `figs/15_hp_selection/fig_hp_<exp>.pdf` — held-out vs in-sample R² per layer and the
  selected profile. The in-sample/held-out divergence is the panel that justifies the new
  rule; it is worth putting in the appendix.

**What to check before trusting a profile**

1. `verification.ranks[*].ceiling_bound` — any `True` means the rank is bound by an
   imposed ceiling rather than chosen; raise `k_cap` / `last_extra` and re-run. (Ignore
   `k_star_equals_kmax`, which is expected: `k_max` is derived from the criterion.)
2. `verification.class_coverage.complete` — `False` means some class has no output factor
   claiming it, which is what constraint 1 exists to prevent.
3. `profile.notes` — states every place the criterion was overridden by a constraint.

**Then:** feed `final_bft_kwargs` into C1 (statistics across seeds) and C2 (pruning). Do
not re-tune anything downstream — that is the point of this notebook.